In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
%run /Workspace/FMCG_Project/consolidated_pipeline/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

base_path = "/Volumes/fmcg/sportsbar-dp/gross_price/*.csv"

### Bronze

In [0]:
df = spark.read.csv(base_path, 
                    header=True, 
                    inferSchema=True
                    )
df = df.withColumn('read_timestamp', F.current_timestamp()) \
       .select("*", "_metadata.file_name", "_metadata.file_size")

display(df.limit(10))

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-04-13T06:02:50.603Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-04-13T06:02:50.603Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-04-13T06:02:50.603Z,gross_price.csv,2741


In [0]:
df.write.format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

In [0]:
display(spark.sql("select * from fmcg.bronze.gross_price"))

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-04-13T06:10:24.127Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-04-13T06:10:24.127Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-04-13T06:10:24.127Z,gross_price.csv,2741


### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source}")
df_bronze.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025/07/01|        -84|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|01/08/2025|    unknown|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025/09/01|         84|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025-10-01|         83|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025-11-01|         83|2026-04-13 06:10:...|gross_price.csv|     2741|
|  88888888|2025-12-01|        -83|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|2025-07-01|         68|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|2025-08-01|         68|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|2025-09-01|         68|2026-04-13 06:10:...|gross_price.csv|     2741|
|  2

In [0]:
df_bronze.select("month").distinct().show()

+----------+
|     month|
+----------+
|2025/07/01|
|01/08/2025|
|2025/09/01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025/11/01|
|2025/08/01|
|01-09-2025|
|2025/10/01|
|01/12/2025|
|01/09/2025|
|01-12-2025|
|01-08-2025|
|01/10/2025|
+----------+



In [0]:
# parse month from multiple possible format
date_formats = ['yyyy/MM/dd', 'dd/MM/yyyy', 'yyyy-MM-dd', 'dd-MM-yyyy']

df_silver = df_bronze.withColumn("month", F.coalesce(
    F.try_to_date(F.col("month"), date_formats[0]),
    F.try_to_date(F.col("month"), date_formats[1]),
    F.try_to_date(F.col("month"), date_formats[2]),
    F.try_to_date(F.col("month"), date_formats[3]),
))

In [0]:
df_silver.select('month').distinct().show()

+----------+
|     month|
+----------+
|2025-07-01|
|2025-08-01|
|2025-09-01|
|2025-10-01|
|2025-11-01|
|2025-12-01|
+----------+



In [0]:
# converting valid gross price values to double. fixing negative prices by making them positive. And replacing all non-numeric values with 0.
df_silver = df_silver.withColumn("gross_price",
            F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'),
                        F.when(F.col("gross_price").cast("double") < 0, -1 * F.col("gross_price").cast("double")).otherwise(F.col("gross_price").cast("double")))
            .otherwise(0)
            )

In [0]:
df_silver.show(10)

+----------+----------+-----------+--------------------+---------------+---------+
|product_id|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+----------+-----------+--------------------+---------------+---------+
|  25891101|2025-07-01|       84.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025-08-01|        0.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025-09-01|       84.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025-10-01|       83.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|2025-11-01|       83.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  88888888|2025-12-01|       83.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|2025-07-01|       68.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|2025-08-01|       68.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|2025-09-01|       68.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  2

In [0]:
# we enrich the silver dataset by performing an inner join with the products table to fetch the correct product code for each product id.

df_products = spark.table("fmcg.silver.products")
df_joined = df_silver.join(df_products.select("product_id", "product_code"), on = "product_id", how="inner")
df_joined = df_joined.select("product_id", "product_code", "month", "gross_price", "read_timestamp", "file_name", "file_size")
df_joined.show(10)

+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|product_id|        product_code|     month|gross_price|      read_timestamp|      file_name|file_size|
+----------+--------------------+----------+-----------+--------------------+---------------+---------+
|  25891101|e91ba9d665f90254d...|2025-07-01|       84.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-08-01|        0.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-09-01|       84.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-10-01|       83.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891101|e91ba9d665f90254d...|2025-11-01|       83.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|e92c739a8d78cd6cb...|2025-07-01|       68.0|2026-04-13 06:10:...|gross_price.csv|     2741|
|  25891102|e92c739a8d78cd6cb...|2025-08-01|       68.0|2026-04-

In [0]:
df_joined.write.format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold

In [0]:
df_silver = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source}")

In [0]:
df_gold = df_silver.select("product_code", "month", "gross_price")
df_gold.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|e91ba9d665f90254d...|2025-07-01|       84.0|
|e91ba9d665f90254d...|2025-08-01|        0.0|
|e91ba9d665f90254d...|2025-09-01|       84.0|
|e91ba9d665f90254d...|2025-10-01|       83.0|
|e91ba9d665f90254d...|2025-11-01|       83.0|
+--------------------+----------+-----------+
only showing top 5 rows


In [0]:
df_gold.write.format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
        .mode("overwrite")\
            .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

### Merging data source with parent

In [0]:
df_gold_price = spark.table("fmcg.gold.sb_dim_gross_price")
df_gold_price.show(5)

+--------------------+----------+-----------+
|        product_code|     month|gross_price|
+--------------------+----------+-----------+
|e91ba9d665f90254d...|2025-07-01|       84.0|
|e91ba9d665f90254d...|2025-08-01|        0.0|
|e91ba9d665f90254d...|2025-09-01|       84.0|
|e91ba9d665f90254d...|2025-10-01|       83.0|
|e91ba9d665f90254d...|2025-11-01|       83.0|
+--------------------+----------+-----------+
only showing top 5 rows


In [0]:
df_gold_price = (
    df_gold_price.withColumn("year", F.year("month")).withColumn("is_zero", F.when(F.col("gross_price")==0, 1).otherwise(0))
)

w = Window.partitionBy("product_code", "year").orderBy(F.col("is_zero"), F.col("month").desc())

df_gold_latest_price = df_gold_price.withColumn("rnk", F.row_number().over(w)).filter(F.col("rnk") == 1)

In [0]:
display(df_gold_latest_price)

product_code,month,gross_price,year,is_zero,rnk
062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379,2025-12-01,281.0,2025,0,1
0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28,2025-10-01,100.0,2025,0,1
102628255d24304d6bbe0438b1ac992054f262e0814d306d0a34d7356cef3268,2025-12-01,86.0,2025,0,1
2e387cef1424d6e7b162b45622d4b1a788d11776e33d05cc8552f4ecd2ea1896,2025-12-01,108.0,2025,0,1
3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345,2025-12-01,493.0,2025,0,1
451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa,2025-12-01,187.0,2025,0,1
716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742,2025-11-01,440.0,2025,0,1
778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6,2025-12-01,296.0,2025,0,1
77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9,2025-12-01,50.0,2025,0,1
889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2,2025-12-01,138.0,2025,0,1


In [0]:
df_gold_latest_price = df_gold_latest_price.withColumnRenamed("gross_price", "price_inr").withColumn("year", F.col("year").cast("string"))
df_gold_latest_price = df_gold_latest_price.select("product_code", "price_inr", "year")

df_gold_latest_price.show(5)

+--------------------+---------+----+
|        product_code|price_inr|year|
+--------------------+---------+----+
|062f5574bbdf4386b...|    281.0|2025|
|0cb7b2f42657b625f...|    100.0|2025|
|102628255d24304d6...|     86.0|2025|
|2e387cef1424d6e7b...|    108.0|2025|
|3cab59f0592428527...|    493.0|2025|
+--------------------+---------+----+
only showing top 5 rows


In [0]:
df_gold_latest_price.createOrReplaceTempView("df_gold_latest_price")

spark.sql(f"""
          MERGE INTO fmcg.gold.dim_gross_price target
          USING df_gold_latest_price SOURCE
          ON target.product_code = source.product_code
          WHEN MATCHED THEN
          UPDATE SET *
          WHEN NOT MATCHED THEN
          INSERT *
          """)

print("Merge completed successfully")

Merge completed successfully


In [0]:
display(spark.sql("select * from fmcg.gold.dim_gross_price"))

product_code,price_inr,year
ARCHDDE20D,2750,2024
ARCH158F41,5740,2024
ARCHAFF0E4,4610,2024
ARCH6B94F7,3910,2024
ARCH5D1FE7,1610,2024
ARCH7B49A9,1610,2024
ARCH497D34,1100,2024
ARCHE71D79,5830,2024
ARCHDD8749,3930,2024
BADMC045D4,4500,2024
